In [1]:
import os, sys

if "COLAB_GPU" in os.environ:
    os.system("git clone https://github.com/thezettascale/fp-emulation.git 2>/dev/null")
    sys.path.insert(0, "/content/fp-emulation/src")
else:
    src = "src" if os.path.isdir("src") else "../src"
    sys.path.insert(0, src)

# DT-PINN: Numerical Derivatives via Matmul

[DT-PINNs](https://arxiv.org/abs/2205.09332) replace autodiff with RBF-FD numerical derivatives. Derivatives become matrix-vector products. No computational graph for higher-order derivatives. Matmul-dominated compute.

In [2]:
import torch
import torch.nn as nn
import time
import os
from pathlib import Path
import matplotlib.pyplot as plt
from fp_emulation import convert

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float64
nu = 0.01 / torch.pi

fig_dir = Path("figures")
if not Path("src").exists():
    fig_dir = Path("..") / "figures"
fig_dir.mkdir(exist_ok=True)

print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## Chebyshev spectral differentiation matrices

Chebyshev collocation ([cheb.m file here](https://people.maths.ox.ac.uk/trefethen/spectral.html)). D1 from Chebyshev nodes, D2 = D1 @ D1. 2D via Kronecker products.

In [3]:
def cheb(N, dtype=torch.float64, device="cpu"):
    """Chebyshev differentiation matrix on N+1 points in [-1, 1]."""
    x = torch.cos(torch.pi * torch.arange(N + 1, dtype=dtype, device=device) / N)
    c = torch.ones(N + 1, dtype=dtype, device=device)
    c[0] = 2.0
    c[N] = 2.0
    c = c * ((-1.0) ** torch.arange(N + 1, dtype=dtype, device=device))
    X = x.unsqueeze(1).expand(N + 1, N + 1)
    dX = X - X.T
    D = (c.unsqueeze(1) / c.unsqueeze(0)) / (dX + torch.eye(N + 1, dtype=dtype, device=device))
    D = D - torch.diag(D.sum(dim=1))
    return x, D


# grid
Nx, Nt = 39, 39  # N+1 = 40 points per dim
x_cheb, Dx = cheb(Nx, dtype, device)
t_cheb, Dt_1d = cheb(Nt, dtype, device)

# rescale t from [-1, 1] to [0, 1]
t_cheb = 0.5 * (t_cheb + 1)
Dt_1d = 2 * Dt_1d  # chain rule

Dxx = Dx @ Dx
Ix = torch.eye(Nx + 1, dtype=dtype, device=device)
It = torch.eye(Nt + 1, dtype=dtype, device=device)

# x-major flattening: kron(Dx, It) for d/dx, kron(Ix, Dt) for d/dt
DX = torch.kron(Dx, It)
DT = torch.kron(Ix, Dt_1d)
DXX = torch.kron(Dxx, It)

# collocation grid
xx, tt = torch.meshgrid(x_cheb, t_cheb, indexing="ij")
X_col = torch.stack([xx.reshape(-1), tt.reshape(-1)], dim=1)

# boundary masks
ic_mask = X_col[:, 1].abs() < 1e-10
bc_mask = ((X_col[:, 0] + 1).abs() < 1e-10) | ((X_col[:, 0] - 1).abs() < 1e-10)

N_total = (Nx + 1) * (Nt + 1)
print(f"Grid: {Nx+1}x{Nt+1} = {N_total} points")
print(f"Diff matrices: {DX.shape}")

Grid: 40x40 = 1600 points
Diff matrices: torch.Size([1600, 1600])


## Burgers' equation: Vanilla PINN vs DT-PINN

$u_t + u \, u_x = \nu \, u_{xx}$, $\nu = 0.01/\pi$. Same problem as `04_demo.ipynb`.

In [4]:
def make_net(dtype):
    return nn.Sequential(
        nn.Linear(2, 256), nn.Tanh(),
        nn.Linear(256, 256), nn.Tanh(),
        nn.Linear(256, 256), nn.Tanh(),
        nn.Linear(256, 1),
    ).to(dtype=dtype, device=device)


def vanilla_loss(model):
    """Derivatives via autograd."""
    x = X_col[:, 0:1].requires_grad_(True)
    t = X_col[:, 1:2].requires_grad_(True)
    u = model(torch.cat([x, t], dim=1))

    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]

    pde = (u_t + u * u_x - nu * u_xx).pow(2).mean()
    ic = (u[ic_mask] + torch.sin(torch.pi * X_col[ic_mask, 0:1])).pow(2).mean()
    bc = u[bc_mask].pow(2).mean()
    return pde + 10 * ic + 10 * bc


def dt_loss(model):
    """Derivatives via matmul (DT-PINN)."""
    u = model(X_col)
    u_x = DX @ u
    u_t = DT @ u
    u_xx = DXX @ u

    pde = (u_t + u * u_x - nu * u_xx).pow(2).mean()
    ic = (u[ic_mask] + torch.sin(torch.pi * X_col[ic_mask, 0:1])).pow(2).mean()
    bc = u[bc_mask].pow(2).mean()
    return pde + 10 * ic + 10 * bc


def sync():
    if device == "cuda":
        torch.cuda.synchronize()


def train(model, loss_fn, n_epochs=500, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(n_epochs):
        loss = loss_fn(model)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())

    return losses


# vanilla PINN (autograd)
torch.manual_seed(0)
model_vanilla = make_net(dtype)
sync()
t0 = time.perf_counter()
losses_vanilla = train(model_vanilla, vanilla_loss)
sync()
time_vanilla = time.perf_counter() - t0

# DT-PINN (FP64 matmul derivatives)
torch.manual_seed(0)
model_dt = make_net(dtype)
sync()
t0 = time.perf_counter()
losses_dt = train(model_dt, dt_loss)
sync()
time_dt = time.perf_counter() - t0

# DT-PINN + Ozaki INT8
torch.manual_seed(0)
model_dt_int8 = convert(make_net(dtype))
sync()
t0 = time.perf_counter()
losses_dt_int8 = train(model_dt_int8, dt_loss)
sync()
time_dt_int8 = time.perf_counter() - t0

print(f"Vanilla PINN (autograd): {time_vanilla:.1f}s")
print(f"DT-PINN (FP64):         {time_dt:.1f}s")
print(f"DT-PINN (INT8 Ozaki):   {time_dt_int8:.1f}s")

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Vanilla PINN (autograd): 25.4s
DT-PINN (FP64):         4.9s
DT-PINN (INT8 Ozaki):   36.2s


In [ ]:
# training time comparison
fig, ax = plt.subplots(figsize=(6, 3))
names = ["Vanilla PINN\n(autograd)", "DT-PINN\n(FP64)", "DT-PINN\n(INT8 Ozaki)"]
times = [time_vanilla, time_dt, time_dt_int8]
colors = ["#4c72b0", "#55a868", "#dd8452"]
ax.bar(names, times, color=colors)
ax.set_ylabel("Training time (s)")
ax.set_title("500 epochs, Burgers' equation")
ax.grid(True, alpha=0.3, axis="y")
for i, t in enumerate(times):
    ax.text(i, t + 0.5, f"{t:.1f}s", ha="center", fontsize=10)
plt.tight_layout()
fig.savefig(fig_dir / "dt_pinn_time.png", dpi=150, bbox_inches="tight")
plt.show()

# loss curves
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(losses_vanilla, label="Vanilla PINN", alpha=0.7)
ax.semilogy(losses_dt, label="DT-PINN FP64", alpha=0.8)
ax.semilogy(losses_dt_int8, "--", label="DT-PINN INT8", alpha=0.8)
ax.set(xlabel="Epoch", ylabel="Loss", title="Burgers: Vanilla PINN vs DT-PINN")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "dt_pinn_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# solution at t=1
x_plot = torch.linspace(-1, 1, 300, dtype=dtype, device=device).unsqueeze(1)
t_plot = torch.ones_like(x_plot)
xt = torch.cat([x_plot, t_plot], dim=1)
with torch.no_grad():
    u_vanilla = model_vanilla(xt).cpu()
    u_dt = model_dt(xt).cpu()
    u_dt_int8 = model_dt_int8(xt).cpu()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_plot.cpu(), u_vanilla, label="Vanilla PINN", lw=2)
ax.plot(x_plot.cpu(), u_dt, label="DT-PINN FP64", lw=2)
ax.plot(x_plot.cpu(), u_dt_int8, "--", label="DT-PINN INT8", lw=2)
ax.set(xlabel="x", ylabel="u", title="Burgers solution at t=1")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "dt_pinn_solution.png", dpi=150, bbox_inches="tight")
plt.show()